# ⛏️ Mineração de Dados
## Crime Data from 2020 to Present - Los Angeles

Este notebook realiza operações de mineração de dados para descobrir padrões e conhecimento.

### Técnicas aplicadas:
1. Análise de Clusters (K-Means, DBSCAN)
2. Redução de Dimensionalidade (PCA)
3. Detecção de Anomalias (Isolation Forest)
4. Classificação de Tipos de Crime
5. Análise de Padrões Temporais

In [ ]:
# Importações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Importar módulos locais
import sys
sys.path.append('..')
from src.mining import *
from src.visualization import *

print('✅ Bibliotecas carregadas com sucesso!')

In [ ]:
# Carregar dados processados
df = pd.read_csv('../data/processed/crime_data_processed.csv')
print(f"Dataset carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()

## 1. Análise de Clusters

Agrupamento de crimes baseado em características geográficas e temporais.

In [ ]:
# Selecionar features para clustering
cluster_features = ['LAT', 'LON', 'HOUR', 'Vict Age']

# Remover valores nulos para as features selecionadas
df_cluster = df[cluster_features].dropna()
print(f"Dados para clustering: {len(df_cluster):,} registros")

In [ ]:
# Método do cotovelo para determinar número ótimo de clusters
print("Análise do método do cotovelo:")
inertias = find_optimal_clusters(df_cluster, cluster_features, max_clusters=10)

In [ ]:
# Plotar método do cotovelo
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), inertias, marker='o')
plt.xlabel('Número de Clusters')
plt.ylabel('Inércia')
plt.title('Método do Cotovelo')
plt.grid(True)
plt.show()

In [ ]:
# Aplicar K-Means com k=5 clusters
df_clustered, kmeans_model = perform_clustering(
    df_cluster, 
    cluster_features, 
    n_clusters=5,
    method='kmeans'
)

# Distribuição dos clusters
print("\nDistribuição dos clusters:")
print(df_clustered['Cluster'].value_counts().sort_index())

In [ ]:
# Aplicar PCA para visualização
X_pca, pca_model = apply_pca(df_cluster, cluster_features, n_components=2)

# Visualizar clusters
fig = plot_cluster_analysis(X_pca, df_clustered['Cluster'].dropna().values)
plt.show()

In [ ]:
# Caracterização dos clusters
cluster_stats = df_cluster.copy()
cluster_stats['Cluster'] = df_clustered['Cluster']

print("Estatísticas por cluster:")
cluster_stats.groupby('Cluster')[cluster_features].mean().round(2)

## 2. Detecção de Anomalias

Identificação de crimes atípicos usando Isolation Forest.

In [ ]:
# Detectar anomalias
anomaly_features = ['LAT', 'LON', 'HOUR', 'Vict Age']

df_anomaly = detect_anomalies(
    df[anomaly_features].dropna(), 
    anomaly_features, 
    contamination=0.05
)

In [ ]:
# Visualizar anomalias geograficamente
plt.figure(figsize=(12, 10))

# Dados normais
normal = df_anomaly[df_anomaly['Is_Anomaly'] == 0]
anomalies = df_anomaly[df_anomaly['Is_Anomaly'] == 1]

plt.scatter(normal['LON'], normal['LAT'], alpha=0.3, s=1, c='blue', label='Normal')
plt.scatter(anomalies['LON'], anomalies['LAT'], alpha=0.5, s=10, c='red', label='Anomalia')

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Distribuição Geográfica de Crimes Normais vs Anomalias')
plt.legend()
plt.show()

In [ ]:
# Características das anomalias
print("Características médias:")
comparison = pd.DataFrame({
    'Normal': df_anomaly[df_anomaly['Is_Anomaly'] == 0][anomaly_features].mean(),
    'Anomalia': df_anomaly[df_anomaly['Is_Anomaly'] == 1][anomaly_features].mean()
})
comparison

## 3. Classificação de Crimes Violentos

In [ ]:
# Preparar dados para classificação
classification_features = ['HOUR', 'Vict Age', 'LAT', 'LON', 'AREA NAME_encoded', 'IS_WEEKEND']
target = 'IS_VIOLENT'

# Verificar disponibilidade das features
available_features = [f for f in classification_features if f in df.columns]
print(f"Features disponíveis: {available_features}")

In [ ]:
# Treinar classificador
if target in df.columns:
    model, report = classify_crime_type(
        df, 
        available_features, 
        target,
        test_size=0.2
    )
    
    print("\nRelatório de Classificação:")
    print(f"Acurácia: {report['accuracy']:.4f}")
    print(f"F1-Score (classe 0): {report['0']['f1-score']:.4f}")
    print(f"F1-Score (classe 1): {report['1']['f1-score']:.4f}")

In [ ]:
# Importância das features
if 'model' in dir():
    importance = get_feature_importance(model, available_features)
    
    plt.figure(figsize=(10, 6))
    plt.barh(importance['Feature'], importance['Importance'])
    plt.xlabel('Importância')
    plt.title('Importância das Features para Classificação de Crime Violento')
    plt.tight_layout()
    plt.show()

## 4. Análise de Padrões Temporais

In [ ]:
# Converter coluna de data
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')

# Análise de padrões
patterns = temporal_pattern_analysis(df.dropna(subset=['DATE OCC']), 'DATE OCC', 'DR_NO')
print("Padrões Temporais:")
patterns

In [ ]:
# Heatmap: Hora vs Dia da Semana
if 'HOUR' in df.columns and 'DAY_OF_WEEK' in df.columns:
    pivot = df.pivot_table(
        values='DR_NO', 
        index='HOUR', 
        columns='DAY_OF_WEEK', 
        aggfunc='count'
    )
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot, cmap='YlOrRd', annot=False)
    plt.xlabel('Dia da Semana (0=Segunda)')
    plt.ylabel('Hora do Dia')
    plt.title('Frequência de Crimes: Hora vs Dia da Semana')
    plt.show()

## 5. Associação entre Área e Tipo de Crime

In [ ]:
# Top 10 tipos de crimes por área
if 'AREA NAME' in df.columns and 'Crm Cd Desc' in df.columns:
    top_crimes = df['Crm Cd Desc'].value_counts().head(10).index
    top_areas = df['AREA NAME'].value_counts().head(10).index
    
    df_filtered = df[df['Crm Cd Desc'].isin(top_crimes) & df['AREA NAME'].isin(top_areas)]
    
    cross_tab = pd.crosstab(df_filtered['AREA NAME'], df_filtered['Crm Cd Desc'])
    
    plt.figure(figsize=(16, 10))
    sns.heatmap(cross_tab, cmap='Blues', annot=True, fmt='d')
    plt.title('Frequência: Área vs Tipo de Crime (Top 10)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 📊 Resumo da Mineração de Dados

### Resultados:
- ✅ Clustering identificou 5 grupos distintos de crimes
- ✅ Detecção de anomalias identificou crimes atípicos
- ✅ Classificador de crimes violentos treinado
- ✅ Padrões temporais identificados
- ✅ Associações entre áreas e tipos de crime mapeadas